# 🏒 NHL Game Data — Phase 1: Exploratory Data Analysis & Data Preparation
**Microsoft Fabric | Junior Data Engineer Final Project**

**Dataset:** [NHL Game Data — Kaggle (martinellis)](https://www.kaggle.com/martinellis/nhl-game-data)  
**Covers:** 6 seasons of NHL games with play-by-play, player stats, goalie stats, and team info

---
## Notebook Overview
| Step | Description |
|------|-------------|
| 1 | Load raw CSV files from Lakehouse |
| 2 | Inspect schema & row counts |
| 3 | Identify & handle nulls, duplicates, data types |
| 4 | EDA — distributions, trends, correlations |
| 5 | Feature engineering |
| 6 | Save cleaned tables back to Lakehouse (Bronze → Silver) |

---
## 1. Setup & Imports

In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')

# Spark is pre-initialised in Fabric — just reference it
spark = SparkSession.builder.getOrCreate()
spark.conf.set('spark.sql.shuffle.partitions', '8')   # optimise for small dataset

print('Spark version:', spark.version)

ModuleNotFoundError: No module named 'pyspark'

---
## 2. Load Raw CSV Files from Lakehouse (Bronze Layer)

In [ ]:
# ── Lakehouse path — update LAKEHOUSE_NAME to your Fabric Lakehouse name ─────
LAKEHOUSE = 'Files/nhl_raw'   # e.g. abfss://... or relative Files/ path in Fabric

def read_csv(filename: str):
    """Read a CSV from the bronze layer with header inference."""
    path = f'{LAKEHOUSE}/{filename}'
    df = spark.read.option('header', True).option('inferSchema', True).csv(path)
    print(f'Loaded {filename}: {df.count():,} rows x {len(df.columns)} cols')
    return df

# Load all tables
game_df            = read_csv('game.csv')
game_plays_df      = read_csv('game_plays.csv')
game_plays_players = read_csv('game_plays_players.csv')
game_shifts_df     = read_csv('game_shifts.csv')
game_skater_stats  = read_csv('game_skater_stats.csv')
game_goalie_stats  = read_csv('game_goalie_stats.csv')
game_teams_stats   = read_csv('game_teams_stats.csv')
player_info_df     = read_csv('player_info.csv')
team_info_df       = read_csv('team_info.csv')

---
## 3. Schema Inspection

In [ ]:
# ── Print schemas for key tables ──────────────────────────────────────────────
for name, df in [
    ('game', game_df),
    ('game_plays', game_plays_df),
    ('game_teams_stats', game_teams_stats),
    ('game_skater_stats', game_skater_stats),
    ('game_goalie_stats', game_goalie_stats),
]:
    print(f'\n=== {name} ===')
    df.printSchema()

---
## 4. Data Quality Assessment

In [ ]:
def null_report(df, name: str):
    """Return a pandas DataFrame showing null counts and percentages."""
    total = df.count()
    null_counts = df.select(
        [F.sum(F.col(c).isNull().cast('int')).alias(c) for c in df.columns]
    ).toPandas().T
    null_counts.columns = ['null_count']
    null_counts['null_pct'] = (null_counts['null_count'] / total * 100).round(2)
    null_counts = null_counts[null_counts['null_count'] > 0].sort_values('null_pct', ascending=False)
    print(f'\n── Null Report: {name} (total rows: {total:,}) ──')
    print(null_counts if not null_counts.empty else 'No nulls found ✅')
    return null_counts

for name, df in [
    ('game', game_df),
    ('game_plays', game_plays_df),
    ('game_teams_stats', game_teams_stats),
    ('game_skater_stats', game_skater_stats),
    ('game_goalie_stats', game_goalie_stats),
    ('player_info', player_info_df),
]:
    null_report(df, name)

In [ ]:
# ── Duplicate check ───────────────────────────────────────────────────────────
def dup_check(df, key_cols: list, name: str):
    total = df.count()
    distinct = df.dropDuplicates(key_cols).count()
    dupes = total - distinct
    print(f'{name}: {dupes:,} duplicate rows on {key_cols} ({dupes/total*100:.2f}%)')

dup_check(game_df, ['game_id'], 'game')
dup_check(game_plays_df, ['play_id'], 'game_plays')
dup_check(game_teams_stats, ['game_id', 'team_id', 'HoA'], 'game_teams_stats')
dup_check(game_skater_stats, ['game_id', 'player_id'], 'game_skater_stats')
dup_check(player_info_df, ['player_id'], 'player_info')
dup_check(team_info_df, ['team_id'], 'team_info')

---
## 5. Exploratory Data Analysis

In [ ]:
# ── 5.1 Games per season ──────────────────────────────────────────────────────
games_per_season = (
    game_df
    .groupBy('season')
    .agg(F.count('game_id').alias('games'),
         F.avg('home_goals').alias('avg_home_goals'),
         F.avg('away_goals').alias('avg_away_goals'))
    .orderBy('season')
    .toPandas()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(games_per_season['season'].astype(str), games_per_season['games'], color='steelblue')
axes[0].set_title('Games per Season')
axes[0].set_xlabel('Season')
axes[0].set_ylabel('Number of Games')
axes[0].tick_params(axis='x', rotation=30)

axes[1].plot(games_per_season['season'].astype(str), games_per_season['avg_home_goals'],
             marker='o', label='Avg Home Goals')
axes[1].plot(games_per_season['season'].astype(str), games_per_season['avg_away_goals'],
             marker='s', label='Avg Away Goals')
axes[1].set_title('Average Goals per Game by Season')
axes[1].set_xlabel('Season')
axes[1].set_ylabel('Avg Goals')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('/tmp/eda_01_season_overview.png', dpi=150)
plt.show()

In [ ]:
# ── 5.2 Home vs Away Win Rate ─────────────────────────────────────────────────
win_stats = (
    game_df
    .groupBy('season', 'type')
    .agg(
        F.sum((F.col('home_goals') > F.col('away_goals')).cast('int')).alias('home_wins'),
        F.sum((F.col('home_goals') < F.col('away_goals')).cast('int')).alias('away_wins'),
        F.count('game_id').alias('total_games')
    )
    .withColumn('home_win_pct', F.round(F.col('home_wins') / F.col('total_games') * 100, 2))
    .orderBy('season', 'type')
    .toPandas()
)

reg = win_stats[win_stats['type'] == 'R']

plt.figure(figsize=(10, 4))
plt.plot(reg['season'].astype(str), reg['home_win_pct'], marker='o', color='darkorange')
plt.axhline(50, linestyle='--', color='gray', alpha=0.6, label='50% line')
plt.title('Home Win % in Regular Season (by Season)')
plt.xlabel('Season')
plt.ylabel('Home Win %')
plt.ylim(40, 65)
plt.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('/tmp/eda_02_home_advantage.png', dpi=150)
plt.show()

In [ ]:
# ── 5.3 Top 10 Scoring Teams (regular season) ─────────────────────────────────
top_teams = (
    game_teams_stats
    .join(team_info_df, 'team_id', 'left')
    .join(game_df.select('game_id', 'type'), 'game_id', 'left')
    .filter(F.col('type') == 'R')
    .groupBy('team_id', 'abbreviation')
    .agg(
        F.sum('goals').alias('total_goals'),
        F.sum('shots').alias('total_shots'),
        F.count('game_id').alias('games_played'),
        F.sum('won').alias('wins')
    )
    .withColumn('goals_per_game', F.round(F.col('total_goals') / F.col('games_played'), 2))
    .withColumn('win_pct', F.round(F.col('wins') / F.col('games_played') * 100, 1))
    .orderBy(F.desc('total_goals'))
    .limit(10)
    .toPandas()
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].barh(top_teams['abbreviation'], top_teams['total_goals'], color='steelblue')
axes[0].set_title('Top 10 Teams — Total Goals (Regular Season)')
axes[0].set_xlabel('Total Goals')
axes[0].invert_yaxis()

axes[1].barh(top_teams['abbreviation'], top_teams['win_pct'], color='seagreen')
axes[1].set_title('Top 10 Teams — Win % (Regular Season)')
axes[1].set_xlabel('Win %')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('/tmp/eda_03_top_teams.png', dpi=150)
plt.show()

In [ ]:
# ── 5.4 Top 20 Skaters by Goals ───────────────────────────────────────────────
top_scorers = (
    game_skater_stats
    .join(player_info_df.select('player_id', 'firstName', 'lastName', 'primaryPosition'), 'player_id', 'left')
    .join(game_df.select('game_id', 'type'), 'game_id', 'left')
    .filter(F.col('type') == 'R')
    .groupBy('player_id', 'firstName', 'lastName', 'primaryPosition')
    .agg(
        F.sum('goals').alias('total_goals'),
        F.sum('assists').alias('total_assists'),
        F.count('game_id').alias('games_played'),
        F.sum('shots').alias('total_shots'),
        F.sum('timeOnIce').alias('total_toi')
    )
    .withColumn('player_name', F.concat(F.col('firstName'), F.lit(' '), F.col('lastName')))
    .withColumn('points', F.col('total_goals') + F.col('total_assists'))
    .withColumn('shooting_pct',
        F.round(F.when(F.col('total_shots') > 0,
                       F.col('total_goals') / F.col('total_shots') * 100).otherwise(0), 1))
    .orderBy(F.desc('total_goals'))
    .limit(20)
    .toPandas()
)

plt.figure(figsize=(12, 7))
bar_colors = ['#e74c3c' if p == 'C' else '#3498db' if p == 'LW' else '#2ecc71' if p == 'RW' else '#f39c12'
              for p in top_scorers['primaryPosition']]
plt.barh(top_scorers['player_name'], top_scorers['total_goals'], color=bar_colors)
plt.title('Top 20 Scorers — Total Goals (Regular Season, All Seasons)')
plt.xlabel('Total Goals')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('/tmp/eda_04_top_scorers.png', dpi=150)
plt.show()

print(top_scorers[['player_name','primaryPosition','total_goals','total_assists','points','shooting_pct','games_played']])

In [ ]:
# ── 5.5 Play-by-Play Event Distribution ──────────────────────────────────────
event_dist = (
    game_plays_df
    .groupBy('event')
    .count()
    .orderBy(F.desc('count'))
    .limit(15)
    .toPandas()
)

plt.figure(figsize=(12, 5))
plt.bar(event_dist['event'], event_dist['count'], color='cornflowerblue')
plt.title('Play-by-Play Event Distribution (Top 15)')
plt.xlabel('Event Type')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.savefig('/tmp/eda_05_event_dist.png', dpi=150)
plt.show()

In [ ]:
# ── 5.6 Shot Heatmap — X/Y coordinates ───────────────────────────────────────
shots_df = (
    game_plays_df
    .filter(F.col('event').isin(['Shot', 'Goal']))
    .select('x', 'y', 'event')
    .dropna()
    .toPandas()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, evt, color in zip(axes, ['Shot', 'Goal'], ['Blues', 'Reds']):
    subset = shots_df[shots_df['event'] == evt]
    ax.hexbin(subset['x'], subset['y'], gridsize=40, cmap=color, mincnt=1)
    ax.set_title(f'{evt} Location Heatmap')
    ax.set_xlabel('X coordinate')
    ax.set_ylabel('Y coordinate')
    ax.axvline(0, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('/tmp/eda_06_shot_heatmap.png', dpi=150)
plt.show()

In [ ]:
# ── 5.7 Goalie Performance — Save % Distribution ─────────────────────────────
goalie_perf = (
    game_goalie_stats
    .join(player_info_df.select('player_id', 'firstName', 'lastName'), 'player_id', 'left')
    .groupBy('player_id', 'firstName', 'lastName')
    .agg(
        F.sum('saves').alias('total_saves'),
        F.sum('shots').alias('total_shots_against'),
        F.count('game_id').alias('games_played')
    )
    .filter(F.col('games_played') >= 20)   # min 20 games for relevance
    .withColumn('save_pct',
        F.round(F.col('total_saves') / F.col('total_shots_against'), 3))
    .withColumn('player_name', F.concat(F.col('firstName'), F.lit(' '), F.col('lastName')))
    .orderBy(F.desc('save_pct'))
    .toPandas()
)

plt.figure(figsize=(10, 4))
plt.hist(goalie_perf['save_pct'].dropna(), bins=30, color='teal', edgecolor='white')
plt.title('Goalie Save % Distribution (Min. 20 Games Played)')
plt.xlabel('Save %')
plt.ylabel('Number of Goalies')
plt.tight_layout()
plt.savefig('/tmp/eda_07_goalie_save_pct.png', dpi=150)
plt.show()

print('Top 10 Goalies by Save %:')
print(goalie_perf[['player_name', 'save_pct', 'games_played', 'total_saves']].head(10))

---
## 6. Data Cleaning & Transformation

In [ ]:
# ── 6.1 Clean game table ──────────────────────────────────────────────────────
game_clean = (
    game_df
    .dropDuplicates(['game_id'])
    .withColumn('date_time', F.to_timestamp('date_time', 'yyyy-MM-dd HH:mm:ss'))
    .withColumn('season_year_start', (F.col('season') / 10000).cast(IntegerType()))
    .withColumn('total_goals', F.col('home_goals') + F.col('away_goals'))
    .withColumn('game_result',
        F.when(F.col('home_goals') > F.col('away_goals'), 'home_win')
         .when(F.col('home_goals') < F.col('away_goals'), 'away_win')
         .otherwise('tie/OT'))
    .na.fill({'outcome': 'unknown', 'home_rink_side_start': 'unknown'})
)
print('game_clean:', game_clean.count(), 'rows')
game_clean.show(3)

In [ ]:
# ── 6.2 Clean game_teams_stats ────────────────────────────────────────────────
teams_stats_clean = (
    game_teams_stats
    .dropDuplicates(['game_id', 'team_id', 'HoA'])
    .withColumnRenamed('HoA', 'home_or_away')
    .withColumn('shot_pct',
        F.round(F.when(F.col('shots') > 0,
                       F.col('goals') / F.col('shots') * 100).otherwise(0), 2))
    .withColumn('faceoff_win_pct',
        F.round(F.when((F.col('faceOffWins') + F.col('faceoffTaken')) > 0,
                       F.col('faceOffWins') / F.col('faceoffTaken') * 100).otherwise(0), 2))
)
print('teams_stats_clean:', teams_stats_clean.count(), 'rows')
teams_stats_clean.show(3)

In [ ]:
# ── 6.3 Clean game_skater_stats ───────────────────────────────────────────────
skater_clean = (
    game_skater_stats
    .dropDuplicates(['game_id', 'player_id'])
    .withColumn('shooting_pct',
        F.round(F.when(F.col('shots') > 0,
                       F.col('goals') / F.col('shots') * 100).otherwise(0), 2))
    .withColumn('points', F.col('goals') + F.col('assists'))
    .withColumn('toi_minutes', F.round(F.col('timeOnIce') / 60, 2))
    .na.fill(0, subset=['goals','assists','shots','hits','blocked','penaltyMinutes',
                        'giveaways','takeaways','plusMinus'])
)
print('skater_clean:', skater_clean.count(), 'rows')
skater_clean.show(3)

In [ ]:
# ── 6.4 Clean game_goalie_stats ───────────────────────────────────────────────
goalie_clean = (
    game_goalie_stats
    .dropDuplicates(['game_id', 'player_id'])
    .withColumn('save_pct',
        F.round(F.when(F.col('shots') > 0,
                       F.col('saves') / F.col('shots')).otherwise(None), 3))
    .withColumn('goals_against_avg',
        F.round(F.when(F.col('timeOnIce') > 0,
                       F.col('goals') / (F.col('timeOnIce') / 3600)).otherwise(None), 2))
    .na.fill(0, subset=['goals', 'saves', 'shots', 'pim', 'evenSaves', 'powerPlaySaves',
                        'shortHandedSaves'])
)
print('goalie_clean:', goalie_clean.count(), 'rows')
goalie_clean.show(3)

In [ ]:
# ── 6.5 Clean game_plays ──────────────────────────────────────────────────────
plays_clean = (
    game_plays_df
    .dropDuplicates(['play_id'])
    .withColumn('period_type',
        F.when(F.col('period') <= 3, 'regulation')
         .when(F.col('period') == 4, 'overtime')
         .otherwise('shootout'))
    .na.fill({'x': 0.0, 'y': 0.0, 'secondaryType': 'N/A', 'description': ''})
)
print('plays_clean:', plays_clean.count(), 'rows')
plays_clean.show(3)

---
## 7. Save Silver Layer Tables to Lakehouse Delta

In [ ]:
# ── Write Silver Delta tables ─────────────────────────────────────────────────
SILVER_PATH = 'Tables/'   # Fabric Lakehouse Delta table path

silver_tables = {
    'silver_game':            game_clean,
    'silver_game_teams_stats': teams_stats_clean,
    'silver_game_skater_stats': skater_clean,
    'silver_game_goalie_stats': goalie_clean,
    'silver_game_plays':      plays_clean,
    'silver_player_info':     player_info_df,
    'silver_team_info':       team_info_df,
}

for table_name, df in silver_tables.items():
    df.write.format('delta').mode('overwrite').option('overwriteSchema', True) \
      .saveAsTable(table_name)
    print(f'✅ Saved: {table_name}')

print('\n🎉 All Silver tables saved to Fabric Lakehouse!')

---
## ✅ Phase 1 Complete

| Table | Status |
|-------|--------|
| silver_game | ✅ Cleaned & enriched |
| silver_game_teams_stats | ✅ Cleaned + shot_pct, faceoff_win_pct |
| silver_game_skater_stats | ✅ Cleaned + points, toi_minutes |
| silver_game_goalie_stats | ✅ Cleaned + save_pct, GAA |
| silver_game_plays | ✅ Cleaned + period_type |
| silver_player_info | ✅ Loaded |
| silver_team_info | ✅ Loaded |

**Next → Run Notebook 02: Gold Layer & Schema Design**